In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

2026-05-29 15:44:49.469069: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780069489.649492      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780069489.696212      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780069490.071312      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780069490.071358      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780069490.071361      58 computation_placer.cc:177] computation placer alr

In [2]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

print("MobileNetV2 Loaded Successfully")

I0000 00:00:1780069562.626561      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
MobileNetV2 Loaded Successfully


In [3]:
dataset_path = "/kaggle/input/datasets/adilmubashirchaudhry/plant-village-dataset/PlantVillageDataset/PlantVillage"

print("Dataset Path Loaded")

Dataset Path Loaded


In [4]:
import os

classes = os.listdir(dataset_path)

print("Number of Classes:", len(classes))
print(classes)

Number of Classes: 16
['PlantVillage', 'Pepper__bell___Bacterial_spot', 'Potato___healthy', 'Tomato_Leaf_Mold', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Bacterial_spot', 'Tomato_Septoria_leaf_spot', 'Tomato_healthy', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato_Early_blight', 'Tomato__Target_Spot', 'Pepper__bell___healthy', 'Potato___Late_blight', 'Tomato_Late_blight', 'Potato___Early_blight', 'Tomato__Tomato_mosaic_virus']


In [5]:
import os

dataset_path = "/kaggle/input/datasets/adilmubashirchaudhry/plant-village-dataset/PlantVillageDataset/PlantVillage"

print(os.listdir(dataset_path))

['PlantVillage', 'Pepper__bell___Bacterial_spot', 'Potato___healthy', 'Tomato_Leaf_Mold', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Bacterial_spot', 'Tomato_Septoria_leaf_spot', 'Tomato_healthy', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato_Early_blight', 'Tomato__Target_Spot', 'Pepper__bell___healthy', 'Potato___Late_blight', 'Tomato_Late_blight', 'Potato___Early_blight', 'Tomato__Tomato_mosaic_virus']


In [6]:
import os

dataset_path = "/kaggle/input/datasets/adilmubashirchaudhry/plant-village-dataset/PlantVillageDataset/PlantVillage/PlantVillage"

classes = os.listdir(dataset_path)

print("Number of Classes:", len(classes))
print(classes)

Number of Classes: 15
['Pepper__bell___Bacterial_spot', 'Potato___healthy', 'Tomato_Leaf_Mold', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Bacterial_spot', 'Tomato_Septoria_leaf_spot', 'Tomato_healthy', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato_Early_blight', 'Tomato__Target_Spot', 'Pepper__bell___healthy', 'Potato___Late_blight', 'Tomato_Late_blight', 'Potato___Early_blight', 'Tomato__Tomato_mosaic_virus']


In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [10]:
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2
)

In [11]:
train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

Found 16516 images belonging to 15 classes.


In [12]:
validation_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 4122 images belonging to 15 classes.


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

In [16]:
model = Sequential([
    base_model,
    
    GlobalAveragePooling2D(),
    
    Dropout(0.5),
    
    Dense(15, activation='softmax')
])

In [18]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [19]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │        19,215 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,277,199 (8.69 MB)

 Trainable params: 19,215 (75.06 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [20]:
from tensorflow.keras.callbacks import EarlyStopping

In [21]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [22]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10


I0000 00:00:1780070716.089139     128 service.cc:152] XLA service 0x7e5b7410f630 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780070716.089199     128 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780070717.221729     128 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780070723.883709     128 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


517/517 ━━━━━━━━━━━━━━━━━━━━ 131s 228ms/step - accuracy: 0.6838 - loss: 0.9904 - val_accuracy: 0.8644 - val_loss: 0.4299
Epoch 2/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 45s 86ms/step - accuracy: 0.8374 - loss: 0.5023 - val_accuracy: 0.8906 - val_loss: 0.3438
Epoch 3/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 45s 87ms/step - accuracy: 0.8559 - loss: 0.4373 - val_accuracy: 0.8984 - val_loss: 0.3096
Epoch 4/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 41s 80ms/step - accuracy: 0.8661 - loss: 0.3981 - val_accuracy: 0.9039 - val_loss: 0.2943
Epoch 5/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 41s 80ms/step - accuracy: 0.8699 - loss: 0.3802 - val_accuracy: 0.9054 - val_loss: 0.2839
Epoch 6/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 43s 83ms/step - accuracy: 0.8752 - loss: 0.3664 - val_accuracy: 0.9129 - val_loss: 0.2697
Epoch 7/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 42s 81ms/step - accuracy: 0.8803 - loss: 0.3507 - val_accuracy: 0.9022 - val_loss: 0.2877
Epoch 8/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 49s 94ms/step - accuracy: 0.8836 - loss: 0.3431 - val_accur

In [23]:
model.save("plant_disease_model.h5")

print("Model Saved Successfully")

Model Saved Successfully


In [24]:
from tensorflow.keras.models import load_model

model = load_model("plant_disease_model.h5")

print("Model Loaded Successfully")

Model Loaded Successfully


In [25]:
class_names = list(train_generator.class_indices.keys())
print(class_names)

['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [26]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = dataset_path + "/Tomato_healthy/your_image.jpg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)
img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/adilmubashirchaudhry/plant-village-dataset/PlantVillageDataset/PlantVillage/PlantVillage/Tomato_healthy/your_image.jpg'

In [27]:
pred = model.predict(img_array)

predicted_class = class_names[np.argmax(pred)]
confidence = np.max(pred)

print("Prediction:", predicted_class)
print("Confidence:", confidence)

NameError: name 'img_array' is not defined

In [28]:
import os

class_folder = dataset_path + "/Tomato_healthy"

print(os.listdir(class_folder)[:10])

['4a1e2b71-992a-4a64-a599-b49b8fa75378___RS_HL 0627.JPG', '7890f4dc-0c55-454b-ab23-3b25d3d973c4___GH_HL Leaf 366.1.JPG', 'beb8ecc5-3283-430e-b598-343f6754a752___GH_HL Leaf 328.JPG', '64b5d77e-2b06-461d-94f9-5f15250a7e77___GH_HL Leaf 505.1.JPG', 'df35d1fb-978c-4aeb-9d6f-e7da242c81b8___RS_HL 0515.JPG', 'a05f8f2d-6ac6-45d8-b7ad-394bbda892fc___RS_HL 0613.JPG', '53471454-9f9a-4182-91a3-c2c6c1268c56___RS_HL 0209.JPG', '93c9da62-19ca-4e90-b355-0dc71ff5cc4b___RS_HL 9752.JPG', 'e70e9a4c-481d-4c9c-8130-aff356057803___RS_HL 9661.JPG', '0de569a8-21c5-4d22-b9c9-ba0ef0f2b614___RS_HL 0314.JPG']


In [30]:
img_path = dataset_path + "/Tomato_healthy/4a1e2b71-992a-4a64-a599-b49b8fa75378___RS_HL 0627.JPG"

In [33]:
import numpy as np
from tensorflow.keras.preprocessing import image

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)
img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)

In [34]:
pred = model.predict(img_array)

predicted_class = class_names[np.argmax(pred)]
confidence = np.max(pred)

print("Prediction:", predicted_class)
print("Confidence:", confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
Prediction: Tomato_healthy
Confidence: 0.99999094
